# Short-Term and Thread Memory with LangGraph

**Short-term memory** is what the agent can see **right now** — typically the message list in graph state. **Thread memory** is that same state **persisted across invocations** so a conversation survives refreshes, crashes, and human handoffs.

LangGraph implements thread memory with **checkpointers**:

```
  invoke #1 (thread_id=alice-42)  ──► checkpoint saved
  invoke #2 (same thread_id)      ──► loads messages + custom state
  invoke   (thread_id=bob-99)     ──► separate checkpoint — no leakage
```

| Concept | LangGraph construct |
|---------|---------------------|
| Message history | `MessagesState` + `add_messages` reducer |
| Thread identity | `configurable.thread_id` |
| Persistence | `MemorySaver` (or Postgres/SQLite in production) |
| Extra scratch fields | Custom `TypedDict` state keys |

This notebook builds **ShopCo Support** as a LangGraph agent with checkpointed thread memory, multi-turn pronoun resolution, thread isolation, and message trimming. Offline by default; optional live LLM at the end.

In [ ]:
"""
ShopCo Support — LangGraph short-term + thread memory lab.
"""

import json
import os
import re
import uuid
from typing import Annotated, Any

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
MAX_MESSAGES_IN_CONTEXT = 8


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


POLICY_SNIPPETS: dict[str, str] = {
    "returns": "Returns allowed within 30 days. Refunds in 5-7 business days. [pol-returns-01]",
    "shipping": "Free shipping over $50. [pol-shipping-02]",
    "billing": "Charges appear as SHOPCO* on statements. [pol-billing-03]",
}

ORDERS_DB: dict[str, dict[str, str]] = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped"},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing"},
}

print("ShopCo support environment ready.")

---

## 1. Short-Term vs Thread Memory

| Term | Meaning | Lifetime |
|------|---------|----------|
| **Short-term** | Current graph state visible to the next node | One super-step / until trimmed |
| **Thread** | Checkpointed state for one conversation ID | Until TTL or explicit delete |
| **Long-term** | Cross-thread user profile (separate store) | Days–months — not covered here |

LangGraph **thread memory** *is* persisted short-term state — messages plus any custom fields you add to `TypedDict`.

---

## 2. Custom State — Messages + Thread Scratch Fields

Extend `MessagesState` with fields the agent updates each turn (`order_id`, `customer_name`). Checkpointer saves them with the thread.

In [ ]:
class SupportThreadState(TypedDict):
    messages: Annotated[list, add_messages]
    customer_name: str
    order_id: str
    turn_count: int
    last_policy_cited: str


def extract_entities_from_message(text: str) -> dict[str, str]:
    updates: dict[str, str] = {}
    m = re.search(r"ORD-[0-9]{4}", text.upper())
    if m:
        updates["order_id"] = m.group(0)
    name_m = re.search(r"my name is ([A-Za-z]+)", text, re.I)
    if name_m:
        updates["customer_name"] = name_m.group(1)
    return updates


print("SupportThreadState defined.")

---

## 3. Graph Nodes — Extract, Respond, Trim

In [ ]:
def extract_node(state: SupportThreadState) -> dict[str, Any]:
    """Pull entities from the latest human message into thread state."""
    last_human = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    if not last_human:
        return {}
    entities = extract_entities_from_message(str(last_human.content))
    return {
        **entities,
        "turn_count": state.get("turn_count", 0) + 1,
    }


def offline_respond_node(state: SupportThreadState) -> dict[str, Any]:
    """Rule-based responder using thread scratch + policy snippets."""
    last_human = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    q = str(last_human.content).lower() if last_human else ""
    name = state.get("customer_name") or "there"
    order_id = state.get("order_id", "")

    if "return" in q or "refund" in q:
        policy = POLICY_SNIPPETS["returns"]
        extra = f" For order {order_id}." if order_id else ""
        answer = f"Hi {name}, {policy}{extra}"
        cited = "pol-returns-01"
    elif "shipping" in q or "delivery" in q:
        answer = f"Hi {name}, {POLICY_SNIPPETS['shipping']}"
        cited = "pol-shipping-02"
    elif "status" in q or ("it" in q and order_id):
        row = ORDERS_DB.get(order_id, {})
        status = row.get("status", "unknown") if row else "not found"
        answer = f"Hi {name}, order {order_id or '(unknown)'} status: {status}."
        cited = ""
    elif "billing" in q or "charge" in q:
        answer = f"Hi {name}, {POLICY_SNIPPETS['billing']}"
        cited = "pol-billing-03"
    else:
        answer = f"Hi {name}, how can I help with your ShopCo order or policy question?"
        cited = ""

    return {"messages": [AIMessage(content=answer)], "last_policy_cited": cited}


def live_respond_node(state: SupportThreadState) -> dict[str, Any]:
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    system = SystemMessage(content=(
        f"ShopCo support. Thread context: customer={state.get('customer_name')}, "
        f"order_id={state.get('order_id')}. Cite policy ids when relevant."
    ))
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}


def respond_node(state: SupportThreadState) -> dict[str, Any]:
    if USE_LIVE_LLM:
        return live_respond_node(state)
    return offline_respond_node(state)


print("Nodes defined: extract → respond")

---

## 4. Build Graph with MemorySaver Checkpointer

In [ ]:
def build_support_graph(checkpointer: MemorySaver | None = None):
    builder = StateGraph(SupportThreadState)
    builder.add_node("extract", extract_node)
    builder.add_node("respond", respond_node)

    builder.add_edge(START, "extract")
    builder.add_edge("extract", "respond")
    builder.add_edge("respond", END)

    return builder.compile(checkpointer=checkpointer)


CHECKPOINTED_GRAPH = build_support_graph(checkpointer=MemorySaver())
STATELESS_GRAPH = build_support_graph(checkpointer=None)

print("Graphs compiled: checkpointed + stateless")

---

## 5. Helper — Send a Turn on a Thread

In [ ]:
def initial_state() -> SupportThreadState:
    return {
        "messages": [],
        "customer_name": "",
        "order_id": "",
        "turn_count": 0,
        "last_policy_cited": "",
    }


def send_turn(
    graph,
    thread_id: str,
    user_text: str,
    *,
    use_checkpoint: bool = True,
) -> SupportThreadState:
    config = {"configurable": {"thread_id": thread_id}} if use_checkpoint else None
    payload = {"messages": [HumanMessage(content=user_text)]}
    if use_checkpoint:
        return graph.invoke(payload, config=config)
    # stateless: must pass full accumulated state manually (shown later)
    return graph.invoke({**initial_state(), **payload})


def last_ai_text(state: SupportThreadState) -> str:
    for m in reversed(state["messages"]):
        if isinstance(m, AIMessage):
            return str(m.content)
    return ""

---

## 6. Multi-Turn Thread — Checkpointed Memory Works

In [ ]:
thread_alice = "support-alice-001"
config_alice = {"configurable": {"thread_id": thread_alice}}

s1 = send_turn(CHECKPOINTED_GRAPH, thread_alice, "My name is Alice. My order is ORD-1001.")
print("Turn 1:", last_ai_text(s1))
print("  state: name=", s1["customer_name"], "order=", s1["order_id"])

s2 = send_turn(CHECKPOINTED_GRAPH, thread_alice, "Can I return it?")
print("\nTurn 2:", last_ai_text(s2))
print("  messages in thread:", len(s2["messages"]))
print("  turn_count:", s2["turn_count"])

s3 = send_turn(CHECKPOINTED_GRAPH, thread_alice, "What is the status?")
print("\nTurn 3:", last_ai_text(s3))
print("  resolved order from thread memory:", s3["order_id"])

---

## 7. Resume After Disconnect — Same thread_id

Simulate the user closing the browser and returning later. Checkpoint reloads message history and scratch fields.

In [ ]:
# Shared checkpointer simulates durable store across graph instances / restarts.
shared_checkpointer = MemorySaver()
graph_a = build_support_graph(checkpointer=shared_checkpointer)
graph_b = build_support_graph(checkpointer=shared_checkpointer)

resume_thread = "support-resume-77"
send_turn(graph_a, resume_thread, "My name is Bob. Order ORD-1002.")

# "Reconnect" on new graph handle, same thread_id
resumed = send_turn(graph_b, resume_thread, "What's the status of my order?")
print("After resume:", last_ai_text(resumed))
print("Thread memory preserved: customer=", resumed["customer_name"], "order=", resumed["order_id"])

---

## 8. Thread Isolation — No Cross-User Leakage

Different `thread_id` values must not share checkpoints.

In [ ]:
send_turn(CHECKPOINTED_GRAPH, "thread-user-a", "My name is Alice. ORD-1001.")
send_turn(CHECKPOINTED_GRAPH, "thread-user-b", "My name is Carol. ORD-1002.")

reply_a = send_turn(CHECKPOINTED_GRAPH, "thread-user-a", "What is my order status?")
reply_b = send_turn(CHECKPOINTED_GRAPH, "thread-user-b", "What is my order status?")

print("Alice thread:", last_ai_text(reply_a))
print("Carol thread:", last_ai_text(reply_b))
assert "ORD-1001" in last_ai_text(reply_a)
assert "ORD-1002" in last_ai_text(reply_b)
print("\nThread isolation OK — no order ID bleed")

---

## 9. Stateless Graph — Why thread_id Alone Is Not Enough

Without a checkpointer, each `invoke` starts fresh unless **you** pass full state back in.

In [ ]:
# Turn 1 — stateless graph forgets everything on turn 2
st1 = STATELESS_GRAPH.invoke({
    **initial_state(),
    "messages": [HumanMessage(content="My name is Alice. ORD-1001.")],
})
st2 = STATELESS_GRAPH.invoke({
    **initial_state(),
    "messages": [HumanMessage(content="Can I return it?")],
})

print("Stateless turn 2 (no memory):")
print(" ", last_ai_text(st2))
print("  order_id in state:", repr(st2.get("order_id")))

---

## 10. Inspecting Checkpoint State

Use `get_state` to debug what the thread remembers between turns.

In [ ]:
inspect_thread = "inspect-demo"
cfg = {"configurable": {"thread_id": inspect_thread}}

CHECKPOINTED_GRAPH.invoke({"messages": [HumanMessage(content="Hi, Alice here. ORD-1001.")]}, config=cfg)
CHECKPOINTED_GRAPH.invoke({"messages": [HumanMessage(content="Return policy?")]}, config=cfg)

snapshot = CHECKPOINTED_GRAPH.get_state(cfg)
values = snapshot.values
print("Checkpoint values:")
print("  customer_name:", values.get("customer_name"))
print("  order_id:", values.get("order_id"))
print("  turn_count:", values.get("turn_count"))
print("  message count:", len(values.get("messages", [])))
print("  next nodes:", snapshot.next)

---

## 11. Short-Term Window — Message Trimming

Long threads exceed context limits. The respond node can **read only the last N messages** while the checkpointer still stores the full thread for audit.

Production options: `trim_messages` helper, summarization node, or `RemoveMessage` with `REMOVE_ALL_MESSAGES` to replace the list safely.

In [ ]:
def context_window(messages: list, limit: int = MAX_MESSAGES_IN_CONTEXT) -> list:
    """Short-term window for model context — full history remains in checkpoint."""
    return messages[-limit:] if len(messages) > limit else messages


long_thread = "long-convo"
for i in range(6):
    state = send_turn(CHECKPOINTED_GRAPH, long_thread, f"Message turn {i+1}: ask about shipping")

full_count = len(state["messages"])
window = context_window(state["messages"])
print(f"Full checkpoint history: {full_count} messages")
print(f"Short-term context window: {len(window)} messages (cap {MAX_MESSAGES_IN_CONTEXT})")
print("Window starts with:", str(window[0].content)[:35], "...")

---

## 12. thread_id Design Guidelines

| Practice | Recommendation |
|----------|------------------|
| **Map to business ID** | Support ticket `TCK-12345` or `user_id + session` |
| **Stability** | Same ticket → same thread across agents |
| **Privacy** | Never reuse thread across different customers |
| **HITL resume** | Human takeover continues same `thread_id` |
| **Production store** | Replace `MemorySaver` with Postgres for durability |

---

## 13. Thread Memory vs Long-Term Memory

| Layer | Scope | LangGraph mechanism |
|-------|-------|---------------------|
| **Short-term / thread** | One conversation | `MessagesState` + checkpointer |
| **Long-term** | Same user, new threads | External store (`BaseStore`), vector user profile |

Thread memory answers *"what did we say in this chat?"* Long-term memory answers *"what do we know about Alice across all chats?"* — typically a separate write path, not the message list alone.

---

## 14. Side-by-Side Comparison

In [ ]:
TURN_PAIR = [
    "My name is Dana. ORD-1002.",
    "Can I return it?",
]


def run_pair(graph, thread: str, checkpointed: bool) -> str:
    for msg in TURN_PAIR:
        state = send_turn(graph, thread, msg, use_checkpoint=checkpointed)
    return last_ai_text(state)


cp_answer = run_pair(CHECKPOINTED_GRAPH, "cmp-cp", True)
sl_answer = run_pair(STATELESS_GRAPH, "cmp-sl", False)

print("Checkpointed (turn 2):", cp_answer[:100], "...")
print("Stateless (turn 2):   ", sl_answer[:100], "...")
print("\nCheckpointed cites order + policy:", "ORD-1002" in cp_answer and "30 days" in cp_answer)

---

## 15. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| No checkpointer | Every invoke forgets thread | `compile(checkpointer=...)` |
| Random thread_id per request | Never multi-turn | Stable ID per ticket/session |
| Shared thread across users | Data leak | One thread per customer session |
| Only messages, no scratch fields | Re-extract every turn | `order_id` in state |
| Unbounded message list | Context overflow | Trim or summarize node |
| MemorySaver in production | State lost on restart | Postgres checkpointer |

---

## 16. Optional Live LLM Respond Node

Set `USE_LIVE_LLM = True` in the setup cell and re-run from graph build onward.

In [ ]:
if USE_LIVE_LLM:
    live_graph = build_support_graph(checkpointer=MemorySaver())
    tid = "live-thread"
    send_turn(live_graph, tid, "I'm Alice, order ORD-1001. What's the return policy?")
    out = send_turn(live_graph, tid, "And status of my order?")
    print(last_ai_text(out))
else:
    print("Offline mode. Example thread state after 2 turns:")
    print(f"  customer={s2['customer_name']} order={s2['order_id']} turns={s2['turn_count']}")
    print("Set USE_LIVE_LLM = True for gpt-4o-mini respond node.")

---

## 17. Quiz

1. What is the difference between short-term state and thread memory in LangGraph?
2. What does `configurable.thread_id` control?
3. Why does a stateless graph fail on "Can I return it?" in turn 2?
4. Why add `order_id` to `TypedDict` instead of only parsing messages each turn?
5. When should you replace `MemorySaver` with a database checkpointer?

---

## 18. Summary

LangGraph **thread memory** = checkpointed graph state keyed by `thread_id`. **Short-term memory** = the messages and scratch fields visible to the next node — often trimmed for context limits.

ShopCo Support demonstrated:

- `SupportThreadState` with `messages` + `order_id` + `customer_name`
- `MemorySaver` for multi-turn and resume
- Thread isolation between customers
- Stateless failure mode vs checkpointed success
- `get_state` for debugging

Use checkpointed threads for conversational agents; use a separate long-term store when facts must survive **new** threads.